
Instead of training from scratch, we adapt powerful pretrained models like
**DistilBERT** using company-specific datasets.


## Why Pretrained Models Are Not Enough

Pretrained models are trained on general data such as:
- Wikipedia
- News articles
- Books

They do NOT understand company-specific language.

### Real Example: HR Resume Screening
A company receives thousands of resumes every month.
A generic model understands English, but not:
- Job-specific keywords
- Resume structure
- Company hiring criteria

Fine-tuning solves this by teaching the model company data.


## What Fine-Tuning Means

Fine-tuning means:
- Taking a pretrained transformer
- Training it again on your labeled dataset

The model already knows language.
We only teach it:
- The task
- The domain

Industry Rule:
Almost all NLP systems in companies are fine-tuned models.


## Install Required Libraries

In [ ]:
!pip install datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00


## Import Required Libraries


In [ ]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

import evaluate


## Why GPU Is Important

Fine-tuning transformers is computationally expensive.

### Industry Reality
Companies always use GPUs to:
- Reduce training time
- Retrain models frequently
- Lower infrastructure cost


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## Loading Business-Like Dataset

All business problems today are **text classification** tasks.

Examples:
- Resume → Job category
- Email → Spam / Not Spam
- Review → Sentiment

We use a public dataset to simulate these tasks.


In [ ]:
dataset = load_dataset("ag_news")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

## Tokenization

Transformers do NOT understand raw text.
They understand numbers.

Tokenization:
- Splits text into tokens
- Converts tokens to IDs
- Adds padding and truncation

Always use the tokenizer that matches the model.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Preprocessing Strategy

Industry-standard choices:
- max_length = 128
- Padding enabled
- Truncation enabled

Shorter sequences = faster training + lower cloud cost.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset["train"].shuffle(seed=42).select(range(10))[0]

{'text': 'Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.',
 'label': 0,
 'input_ids': [101,
  7269,
  11498,
  2135,
  6924,
  2011,
  9326,
  4559,
  10134,
  2031,
  2716,
  2116,
  4865,
  1998,
  3655,
  1999,
  7269,
  2000,
  1037,
  9190,
  1010,
  1996,
  2154,
  2044,
  2324,
  2111,
  2351,
  1999,
  18217,
  2012,
  1037,
  2576,
  8320,
  1012,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,

## Train and Evaluation Split

Why split data?
- Training data → learning
- Evaluation data → performance measurement

Never evaluate on training data (data leakage).


In [ ]:
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
eval_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))


## Choosing DistilBERT

Why DistilBERT?
- Smaller than BERT
- Faster inference
- Near-BERT accuracy

Industry prefers speed + performance.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=4
).to(device)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Evaluation Metrics

Accuracy alone is misleading.

Industry metrics:
- Precision
- Recall
- F1-score

Example:
Spam detection needs high recall and precision.


In [ ]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels,)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="weighted")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="weighted")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="weighted")["f1"],
    }


## Hugging Face Trainer API

Trainer handles:
- Training loop
- GPU usage
- Evaluation
- Logging
- Model saving

Companies do not write training loops manually.


In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_steps=500,          # evaluate every 500 steps
    logging_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    save_steps=500,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-804189787.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## Fine-Tuning the Model

During training:
- Loss decreases
- Model adapts to business data
- Metrics improve


In [ ]:
trainer.train()


Step,Training Loss
500,0.402300


TrainOutput(global_step=626, training_loss=0.3696927079758324, metrics={'train_runtime': 136.8488, 'train_samples_per_second': 73.073, 'train_steps_per_second': 4.574, 'total_flos': 331180308480000.0, 'train_loss': 0.3696927079758324, 'epoch': 2.0})

In [ ]:
trainer.evaluate()


{'eval_loss': 0.3158605396747589,
 'eval_accuracy': 0.898,
 'eval_precision': 0.8996132511690819,
 'eval_recall': 0.898,
 'eval_f1': 0.897932366225813,
 'eval_runtime': 3.6493,
 'eval_samples_per_second': 274.029,
 'eval_steps_per_second': 17.264,
 'epoch': 2.0}

## Saving Model for Production

Saved models are:
- Deployed via APIs
- Version-controlled
- Business assets


In [ ]:
trainer.save_model("./fine_tuned_distilbert")
tokenizer.save_pretrained("./fine_tuned_distilbert")


('./fine_tuned_distilbert/tokenizer_config.json',
 './fine_tuned_distilbert/special_tokens_map.json',
 './fine_tuned_distilbert/vocab.txt',
 './fine_tuned_distilbert/added_tokens.json',
 './fine_tuned_distilbert/tokenizer.json')

## Loading the Fine-Tuned Model (Production Use)

After training, models are usually:
- Saved to disk
- Loaded later for inference
- Used inside APIs (Flask / FastAPI)

This step simulates **real production usage** where training
and inference happen at different times.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "./fine_tuned_distilbert"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.to(device)
model.eval()


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### Why do we call `model.eval()`?

- Disables dropout
- Ensures stable predictions
- Required for inference

In production, models are **always** in evaluation mode.


## Using the Model for Real Business Predictions

Now we will use the fine-tuned model to classify new,
unseen text — exactly what happens in a real application.

### Example Scenarios
- Resume classification
- Spam email detection
- Review sentiment analysis


In [ ]:
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

    return predicted_class


### Understanding Model Output

This dataset uses the following labels:

| Label ID | Meaning |
|--------|--------|
| 0 | World |
| 1 | Sports |
| 2 | Business |
| 3 | Sci/Tech |

In a real company:
- These would be resume categories, spam labels, or sentiment classes.


In [ ]:
#Example 1: News / Resume Classification
text_1 = "Experienced Python developer with machine learning and cloud experience"
prediction = predict(text_1)

print("Predicted label:", prediction)


Predicted label: 3


### Interpretation

The model predicts which category the text belongs to.

In a real HR system:
- Label → Job category
- Next step → route resume to recruiter


In [ ]:
#Example 2: Spam Email Detection
text_2 = "Congratulations! You have won a free prize. Click here now!"
prediction = predict(text_2)

print("Predicted label:", prediction)


Predicted label: 3


This simulates how companies classify customer feedback automatically.


In [ ]:
text_3 = "The delivery was late and the product quality is very poor"
prediction = predict(text_3)

print("Predicted label:", prediction)

Predicted label: 2


## Why the Model Predicts Numbers

Machine learning models do not understand text labels.
They work with **numerical class IDs** internally.

Example:
- Model output → `2`
- Human meaning → `"Business"`

To make predictions useful, we must map **label IDs → label names**.


## Label Mapping

For our dataset, labels mean:

| ID | Label |
|----|-------|
| 0 | World |
| 1 | Sports |
| 2 | Business |
| 3 | Sci/Tech |

In real projects:
- These could be job roles, spam categories, or sentiment classes.


In [ ]:

label_names = dataset["train"].features["label"].names


label_map = {i: name for i, name in enumerate(label_names)}

label_map

{0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

In [ ]:
def predict_with_label(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_id = torch.argmax(logits, dim=1).item()
    predicted_label = label_map[predicted_id]

    return predicted_id, predicted_label

In [ ]:
text = "Experienced Python developer with machine learning background"
pred_id, pred_label = predict_with_label(text)

print(f"Predicted ID: {pred_id}")
print(f"Predicted Label: {pred_label}")

Predicted ID: 3
Predicted Label: Sci/Tech


# Task: Building a Real Spam Classifier

**Step 1: Explore the Dataset**
- Load the `sms_spam` dataset from Hugging Face.
- Check column names and label names.
- Expected:
  - Text column: `text`
  - Label column: `label`
  - Labels: `['ham', 'spam']`

**Step 2: Create Label Dictionary**
- Map ID → Label name and Label name → ID.
```python
label_map = {0: 'ham', 1: 'spam'}
id_map = {'ham': 0, 'spam': 1}


**Step 3: Tokenize and Preprocess**
- Use AutoTokenizer from distilbert-base-uncased.
- Apply padding, truncation, and max_length=128.

**Step 4: Split Train & Evaluation Data**
- Shuffle dataset.
- Use 5000 samples for training and 1000 for evaluation.

**Step 5: Fine-Tune DistilBERT**
- Load AutoModelForSequenceClassification with num_labels=2.
- Use Trainer API with metrics: Accuracy, Precision, Recall, F1.
- Suggested hyperparameters: learning_rate=2e-5, batch_size=16, epochs=2.

**Step 6: Save Model**
```python
trainer.save_model("./spam_model")
tokenizer.save_pretrained("./spam_model")

**Step 7: Load Model & Make Predictions**
- Load model & tokenizer.
- Create a prediction function returning human-readable labels.
Test examples:
```python
texts = [
    "Congratulations! You've won a free ticket.",
    "Hey, are we meeting tomorrow?",
]

for t in texts:
    print(predict_with_label(t))